# Exercise 1.3 — Build a Word Frequency Counter
### *Discover that "actually" is doing a lot of unpaid overtime*

Ever sat in a meeting and thought "did we really need to say 'actually' forty times in twenty minutes"? Today you're going to prove it with data, not vibes. You'll build a small tool that reads any block of text — a transcript, an email thread, a document — and tells you exactly which words show up most, and how often.

**Why learn this instead of pasting the transcript into an AI chatbot?**

You could do that, and it would work fine for a one-off. But once you understand how text gets broken into words and counted, you'll know how to run this on *any* text, offline, instantly, for free, without pasting potentially confidential meeting notes into a third-party tool. That's a real, practical reason to know this — not just a coding-class exercise.

## What you'll practise
- **Strings** — cleaning and splitting text
- **Loops** — going word by word
- **Sorting** — putting the results in order, most-used first

## The scenario
`meeting_transcript.txt` (generated for you, next to this notebook) contains a short, slightly exaggerated meeting transcript. Let's find out what word this team can't stop saying.

## Step 1: Read the file and look at the raw text

In [1]:
with open("meeting_transcript.txt") as f:
    text = f.read()

print(text[:400])
print("...")
print(f"\nTotal characters: {len(text)}")


Alex: Okay so, actually, I think the main issue is that the dashboard is actually loading really slowly for clients.
Priya: Right, right. So actually, we looked into it yesterday and it's actually the database queries that are the bottleneck.
Marcus: Can we actually just add an index to that table? That should actually speed things up quite a bit.
Alex: Yeah, actually that's a good point. Let's ac
...

Total characters: 1632


## Step 2: Turn the text into a list of words

This is the trickiest part conceptually, so let's go slow. If we just do `text.split()`, we get words — but punctuation sticks to them (`"clients."` is not the same string as `"clients"`), and capitalisation makes `"Actually"` and `"actually"` count as different words. We need to clean that up first.

In [2]:
# First, the naive way - let's see the problem
naive_words = text.split()
print("First 10 words, naive split:")
print(naive_words[:10])


First 10 words, naive split:
['Alex:', 'Okay', 'so,', 'actually,', 'I', 'think', 'the', 'main', 'issue', 'is']


In [3]:
import string

# Lowercase everything, then strip punctuation off each word
def clean_words(text):
    text = text.lower()
    words = text.split()
    cleaned = []
    for word in words:
        # str.strip(chars) removes any of those characters from the start/end
        word = word.strip(string.punctuation)
        if word:  # skip anything that becomes empty (like a lone "-")
            cleaned.append(word)
    return cleaned

words = clean_words(text)
print("First 10 words, cleaned:")
print(words[:10])
print(f"\nTotal word count: {len(words)}")


First 10 words, cleaned:
['alex', 'okay', 'so', 'actually', 'i', 'think', 'the', 'main', 'issue', 'is']

Total word count: 273


**What just happened?**

- `text.lower()` makes `"Actually"` and `"actually"` identical, so they'll count as the same word.
- `word.strip(string.punctuation)` removes punctuation *only from the edges* of each word — so `"clients."` becomes `"clients"`, but something like `"don't"` keeps its apostrophe in the middle. `string.punctuation` is a handy built-in string containing every common punctuation character, so we don't have to type them all out ourselves.

## Step 3: Count each word using a dictionary

In [4]:
word_counts = {}

for word in words:
    if word in word_counts:
        word_counts[word] += 1
    else:
        word_counts[word] = 1

# Let's check our word of interest
print("Count of 'actually':", word_counts.get("actually", 0))
print("Count of 'the':", word_counts.get("the", 0))


Count of 'actually': 31
Count of 'the': 10


**Key idea:** this loop is one of the most common patterns in all of programming — "count how many times each thing appears." A dictionary is perfect for it: the word is the *key*, and the running count is the *value*. Every time we see a word again, we bump its count up by one.

There's actually a shortcut for this exact pattern built into Python — `collections.Counter` — but building it manually once, like we just did, means you'll always understand what's happening underneath any shortcut you use later.

## Step 4: Sort by frequency and show the top words

In [5]:
# word_counts.items() gives us (word, count) pairs
# sorted(..., key=..., reverse=True) sorts them by count, highest first
sorted_words = sorted(word_counts.items(), key=lambda pair: pair[1], reverse=True)

print(f"{'WORD':<15}{'COUNT'}")
print("-" * 25)
for word, count in sorted_words[:15]:
    print(f"{word:<15}{count}")


WORD           COUNT
-------------------------
actually       31
the            10
a              9
we             8
that           6
alex           5
is             5
it             5
this           5
so             4
priya          4
and            4
it's           4
let's          4
okay           3


There it is, in black and white. Somebody's sending this meeting transcript to the whole team.

**What's `key=lambda pair: pair[1]` doing?** `sorted()` needs to know *what part* of each `(word, count)` pair to sort by. `lambda pair: pair[1]` is a tiny, throwaway function that says "given a pair, give me its second item (the count)." It's a shortcut for writing a full function just for this one-off use.

## Step 5: Filter out boring filler words (optional polish)

In [6]:
# Common words like "the", "is", "and" will always dominate any English text.
# In real text analysis these are called "stop words" - let's filter a few out.
stop_words = {"the", "a", "an", "is", "it", "to", "and", "of", "so", "we", "i", "this",
              "that", "for", "on", "in", "with", "let's", "let", "one"}

meaningful_words = {word: count for word, count in word_counts.items() if word not in stop_words}
sorted_meaningful = sorted(meaningful_words.items(), key=lambda pair: pair[1], reverse=True)

print(f"{'WORD':<15}{'COUNT'}")
print("-" * 25)
for word, count in sorted_meaningful[:10]:
    print(f"{word:<15}{count}")


WORD           COUNT
-------------------------
actually       31
alex           5
priya          4
it's           4
okay           3
right          3
marcus         3
index          3
should         3
that's         3


That line uses a **dictionary comprehension** — `{word: count for word, count in word_counts.items() if word not in stop_words}` — which is just a compact way of writing "build a new dictionary, keeping only the entries that pass this condition." It's the same idea as a loop with an `if` check inside, just written in one line once you're comfortable with the pattern.

## 🎉 You did it

You now have a reusable tool: point it at any transcript, email thread, or document, and it'll tell you the most common words in seconds — no uploading anything anywhere. And you understand *why* it works, not just that it does.

---

## 🧪 Try it yourself

1. Run this on a document of your own — paste different text into a `.txt` file and re-run the notebook.
2. Modify the code to count *phrases* of two words instead of single words (e.g. "actually a" appearing 3 times). Hint: instead of looping through `words`, loop through pairs — `words[i]` and `words[i+1]`.
3. Add a bar chart of the top 10 words using `matplotlib` (`import matplotlib.pyplot as plt`) so it's a visual, not just a table.

## 💡 Where this goes next
This is the exact same idea behind "word cloud" generators and basic text-analytics dashboards. You just built the engine underneath one, from scratch.